# Workshop Lite: AgentCore + LangGraph ReAct (Google Docs)

Спростив флоу для демо:

- мінімум інфра-коду на екрані;
- офіційні патерни AgentCore + boto3 для control plane;
- HTTP invoke для runtime OAuth flow;
- складна логіка схована у `workshop_helpers.WorkshopE2EDemo`.

## Prerequisites

У `.env` мають бути:

- `GOOGLE_CLIENT_ID`
- `GOOGLE_CLIENT_SECRET`
- `GOOGLE_DOC_ID`
- `GOOGLE_API_KEY`

Опціонально:

- `AWS_PROFILE` (default: `workshop`)
- `AWS_REGION` (default: `us-east-1`)
- `WORKSHOP_PREFIX` (default: `acwslite`)
- `OAUTH_RETURN_URL` (default: `http://localhost:8081/oauth2/callback`)

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🛠 Що відбувається у цій клітинці:</strong><br>
  Ініціалізуємо demo helper, завантажуємо `.env`, створюємо AWS session і збираємо стандартизовані імена ресурсів для воркшопу.
</div>

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / 'runtime_app_agentcore_full.py').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Could not locate project root from the notebook directory.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from workshop_helpers import WorkshopE2EDemo

demo = WorkshopE2EDemo(PROJECT_ROOT)
print(json.dumps(demo.summary(), indent=2))

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔍 Що відбувається у цій клітинці:</strong><br>
  Робимо простий preflight через `boto3 STS`, щоб переконатися, що права доступу та профіль AWS працюють коректно.
</div>

In [ ]:
caller = demo.sts.get_caller_identity()
print(json.dumps(caller, indent=2))

## Step 1 - Inbound auth (Cognito)

Створюємо/перевикористовуємо:

- User Pool
- App Client (`USER_PASSWORD_AUTH`)
- demo user

Отримуємо `access_token` для inbound JWT викликів.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔑 Що відбувається у цій клітинці:</strong><br>
  Створюємо або знаходимо існуючий Cognito User Pool, client app і demo user. Наприкінці отримуємо `access_token`, який піде в inbound JWT flow.
</div>

In [ ]:
inbound = demo.ensure_inbound_auth()
user_access_token = demo.state['user_access_token']

print(json.dumps(inbound, indent=2))
print('access_token_length:', len(user_access_token))

## Step 2 - Outbound provider + Gateway

1. Створюємо OAuth provider (Google) в AgentCore Identity.
2. Створюємо Gateway з inbound JWT (Cognito discovery + allowed client).
3. Додаємо target з Google Docs OpenAPI + OAuth credential provider.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔗 Що відбувається у цій клітинці:</strong><br>
  Реєструємо Google OAuth provider в AgentCore Identity. Це створює callback URL, який треба додати в Google OAuth client redirect URIs.
</div>

In [ ]:
provider = demo.ensure_google_provider()
print(json.dumps(provider, indent=2))
print('Add callback_url to Google OAuth redirect URIs if it is not there yet.')

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🌐 Що відбувається у цій клітинці:</strong><br>
  Створюємо або перевикористовуємо AgentCore Gateway з JWT authorizer, який довіряє нашому Cognito User Pool та клієнту.
</div>

In [ ]:
gateway = demo.ensure_gateway()
print(json.dumps(gateway, indent=2))

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🧩 Що відбувається у цій клітинці:</strong><br>
  Підключаємо до Gateway Google Docs target через OpenAPI schema та OAuth credential provider. Одразу робимо smoke-test шляху `tools/call`.
</div>

In [ ]:
gateway = demo.ensure_google_docs_target()
demo.health_gate()
smoke = demo.smoke_test_gateway()

print(json.dumps(gateway, indent=2))
print('gateway_smoke_status:', smoke['status'])
if smoke.get('authorization_url'):
    print('gateway_authorization_url:', smoke['authorization_url'])

## Step 3 - Local ReAct Agent (Mock Data)

Цей крок навмисно ізольований від AWS/Gateway OAuth, щоб локальне навчання було стабільним.

Що тут є:

- `ChatGoogleGenerativeAI` як LLM
- `@tool` + `create_react_agent`
- Один мок-інструмент: `get_workshop_doc`
- Локальний invoke без зовнішнього OAuth

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🤖 Що відбувається у цій клітинці:</strong><br>
  Створюємо локального LangGraph ReAct агента з мок-документом, щоб окремо показати reasoning/tool-calling без AWS інфраструктури.
</div>

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model=demo.google_model_id,
    google_api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)

MOCK_DOC_SOURCE = 'https://docs.google.com/document/d/mock-incident-response-workshop/edit'
MOCK_DOC_TEXT = """
Incident response process in this workshop document:

1. Detect and triage incidents within 15 minutes.
2. Assign incident commander and communication owner.
3. Contain impact first, then analyze root cause.
4. Publish post-incident review with action items within 48 hours.
""".strip()

@tool
def get_workshop_doc() -> str:
    """Return mock workshop document text about incident response."""
    return f"""DOCUMENT_TEXT:
{MOCK_DOC_TEXT}

SOURCE: {MOCK_DOC_SOURCE}"""

local_agent = create_react_agent(
    model=llm,
    tools=[get_workshop_doc],
    prompt='Call get_workshop_doc first. Answer only from document evidence and include Sources.',
    checkpointer=InMemorySaver(),
)

print('Local mock agent is ready.')

In [ ]:
def print_local_trajectory(result_dict):
    print('🤖 Локальний агент (Траєкторія):\n' + '-' * 60)
    for msg in result_dict['messages']:
        role = getattr(msg, 'type', type(msg).__name__)
        content = msg.content
        blocks = []
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict) and 'text' in block:
                    blocks.append(block['text'])
        elif isinstance(content, str):
            blocks.append(content)
        text = '\n'.join(blocks).strip()
        if text:
            print(f'[{role.upper()}]\n{text}\n')
        if getattr(msg, 'tool_calls', None):
            for tc in msg.tool_calls:
                print(f"Tool call: {tc['name']} args={tc['args']}\n")

local_result = local_agent.invoke(
    {'messages': [('user', 'What does this document say about incident response?')]},
    config={'configurable': {'thread_id': 'local-react-demo-0001'}},
)
print_local_trajectory(local_result)

## Step 4 - Deploy runtime through AgentCore SDK

Runtime entrypoint: `runtime_app_agentcore_full.py` (офіційний pattern з `BedrockAgentCoreApp`).

У notebook лишається лише один виклик `deploy_runtime()`, а packaging, S3 upload, IAM role та create/update runtime сховані в helper.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🚀 Що відбувається у цій клітинці:</strong><br>
  Пакуємо runtime код, заливаємо artifact у S3, створюємо або оновлюємо AgentCore Runtime через SDK і чекаємо статус `READY`.
</div>

In [ ]:
runtime = demo.deploy_runtime()
print(json.dumps(runtime, indent=2))

## Step 5 - Runtime invoke + consent

1. У notebook явно формуємо runtime payload для першого invoke.
2. Перший invoke повертає consent URL, якщо Google OAuth ще не прив'язаний до сесії.
3. Після consent окремо формуємо другий payload і повторюємо invoke.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>📞 Що відбувається у цій клітинці:</strong><br>
  Підіймаємо локальний callback server для OAuth redirect, явно формуємо runtime payload і робимо перший invoke. Якщо consent ще не виданий, побачимо `authorization_url` та `oauth_session_uri`.
</div>

In [ ]:
callback_server = demo.start_callback_server()
print(json.dumps(callback_server, indent=2))

first_payload = demo.build_runtime_payload(
    prompt='Summarize incident response from this document in 6 bullets and include source.',
    max_steps=5,
    max_doc_calls=1,
)
print(json.dumps(first_payload, indent=2))

first = demo.invoke_runtime(payload=first_payload)
demo.print_runtime_result('FIRST INVOKE', first)
demo.assert_runtime_version(first)

print('authorization_url:', (first.get('authorization_url') or '').strip() or '<not required>')
print('oauth_session_uri:', (first.get('oauth_session_uri') or '').strip() or '<not returned>')

> Якщо consent ще не виданий, відкрий `authorization_url`, заверши Google login/consent і повернись у notebook.

> Тепер локальний callback server уже запущений, тому після Google consent ти маєш побачити сторінку `Consent complete`, а не помилку з'єднання.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🔄 Що відбувається у цій клітинці:</strong><br>
  Після consent завершуємо OAuth session у AgentCore token vault, явно формуємо другий payload і робимо повторний invoke, який вже повертає фінальну відповідь агента.
</div>

In [ ]:
demo.complete_runtime_consent(first.get('oauth_session_uri'))

second_payload = demo.build_runtime_payload(
    prompt='Answer in 6 bullets: summarize incident response from the document and cite source link from the document.',
    thread_id=first.get('thread_id'),
    max_steps=5,
    max_doc_calls=1,
)
print(json.dumps(second_payload, indent=2))

second = demo.invoke_runtime(payload=second_payload)
demo.print_runtime_result('SECOND INVOKE', second)
demo.assert_runtime_version(second)

## Step 6 - Cleanup

Опціональний крок для видалення воркшопних ресурсів. Використовуй лише коли хочеш повністю прибрати створені Cognito, AgentCore Identity, Gateway і Runtime ресурси.

<div style="background-color: #f8fbff; padding: 12px 16px; border-left: 4px solid #0073e6; border-radius: 4px; margin-bottom: 8px; font-family: sans-serif;">
  <strong>🗑️ Що відбувається у цій клітинці:</strong><br>
  Видаляємо створені під час воркшопу ресурси. Логіка cleanup схована в helper, щоб не захаращувати notebook.
</div>

In [ ]:
# demo.cleanup()